# NumPy Performance Analytics

### An End-to-End Employee Analytics Project Built Entirely with NumPy

This notebook applies core NumPy concepts — array creation, vectorized arithmetic, boolean indexing, aggregation, sorting, and performance benchmarking — to a realistic employee analytics dataset.


## 2. Objective

The goal of this notebook is to analyze a synthetic employee dataset of **100,000 records** using only **NumPy** and Python's built-in **`time`** module.

We will:

- Generate synthetic employee data using NumPy's random and array-creation functions.
- Compute salary, attendance, department, and performance statistics using vectorized operations.
- Calculate bonuses and final salaries without using explicit Python loops.
- Build a consolidated final report using `np.column_stack()`.
- Benchmark a traditional Python loop against a NumPy vectorized operation to quantify the performance gain.

No Pandas, Matplotlib, Seaborn, or SciPy are used — every computation relies purely on NumPy arrays and operations.


## 3. Import Libraries

Only two imports are required for this entire project:

- **`numpy`** — for array creation, vectorized math, and statistical functions.
- **`time`** — for measuring execution time in the performance benchmark section.


In [ ]:
import numpy as np
import time

print(f"NumPy version: {np.__version__}")


## 4. Generate Synthetic Employee Dataset

We generate data for **100,000 employees** using several NumPy array-creation and random-sampling functions:

- `np.arange()` — to generate sequential, unique employee IDs.
- `np.random.choice()` — to randomly assign each employee to one of six departments.
- `np.random.uniform()` — to generate continuous salary values within a realistic range.
- `np.random.randint()` — to generate integer attendance percentages, overtime hours, and performance scores.

`np.random.seed(42)` is set first so the dataset is **reproducible** — every run produces identical results.


In [ ]:
np.random.seed(42)

NUM_EMPLOYEES = 100_000

departments_list = ["Engineering", "HR", "Finance", "Sales", "Marketing", "Operations"]

# Sequential unique employee IDs starting at 1
employee_id = np.arange(1, NUM_EMPLOYEES + 1)

# Random department assignment for each employee
department = np.random.choice(departments_list, size=NUM_EMPLOYEES)

# Continuous salary values between 30,000 and 150,000
salary = np.random.uniform(30_000, 150_000, size=NUM_EMPLOYEES).round(2)

# Attendance percentage between 60 and 100
attendance = np.random.randint(60, 101, size=NUM_EMPLOYEES)

# Overtime hours worked per month, between 0 and 40
overtime_hours = np.random.randint(0, 41, size=NUM_EMPLOYEES)

# Performance score between 50 and 100
performance_score = np.random.randint(50, 101, size=NUM_EMPLOYEES)

print("Dataset generated successfully.")
print(f"Total employees: {NUM_EMPLOYEES:,}")


## 5. Dataset Overview

Before analysis, we inspect the shape, data types, and a preview of each array. This confirms the dataset was constructed correctly.


In [ ]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"{'Array':<20}{'Shape':<15}{'Dtype':<10}")
print("-" * 60)
for name, arr in [
    ("employee_id", employee_id),
    ("department", department),
    ("salary", salary),
    ("attendance", attendance),
    ("overtime_hours", overtime_hours),
    ("performance_score", performance_score),
]:
    print(f"{name:<20}{str(arr.shape):<15}{str(arr.dtype):<10}")

print("\nFirst 5 records:")
print("-" * 60)
for i in range(5):
    print(
        f"ID {employee_id[i]:>6} | {department[i]:<12} | "
        f"Salary ${salary[i]:>10,.2f} | Attendance {attendance[i]:>3}% | "
        f"Overtime {overtime_hours[i]:>2}h | Performance {performance_score[i]:>3}"
    )


## 6. Salary Analytics

**What:** Compute central-tendency and spread statistics for the `salary` array.

**Why:** These metrics summarize compensation distribution at a glance — useful for HR budgeting and compensation benchmarking.

**NumPy concepts:** `np.mean()`, `np.median()`, `np.max()`, `np.min()`, `np.std()`.


In [ ]:
avg_salary = np.mean(salary)
median_salary = np.median(salary)
max_salary = np.max(salary)
min_salary = np.min(salary)
std_salary = np.std(salary)

print("=" * 60)
print("SALARY ANALYTICS")
print("=" * 60)
print(f"Average Salary     : ${avg_salary:,.2f}")
print(f"Median Salary       : ${median_salary:,.2f}")
print(f"Highest Salary      : ${max_salary:,.2f}")
print(f"Lowest Salary       : ${min_salary:,.2f}")
print(f"Standard Deviation  : ${std_salary:,.2f}")


## 7. Attendance Analytics

**What:** Summarize attendance percentages and identify employees at the low and high ends of attendance.

**Why:** Attendance thresholds help HR flag employees who may need support (below 85%) and recognize strong attendance (above 95%).

**NumPy concepts:** `np.mean()`, `np.max()`, `np.min()`, and **boolean indexing** (`array[condition]`) to filter and count employees meeting a condition.


In [ ]:
avg_attendance = np.mean(attendance)
max_attendance = np.max(attendance)
min_attendance = np.min(attendance)

# Boolean indexing to filter employees by attendance thresholds
below_85_mask = attendance < 85
above_95_mask = attendance > 95

employees_below_85 = np.sum(below_85_mask)
employees_above_95 = np.sum(above_95_mask)

print("=" * 60)
print("ATTENDANCE ANALYTICS")
print("=" * 60)
print(f"Average Attendance      : {avg_attendance:.2f}%")
print(f"Maximum Attendance      : {max_attendance}%")
print(f"Minimum Attendance      : {min_attendance}%")
print(f"Employees Below 85%     : {employees_below_85:,}")
print(f"Employees Above 95%     : {employees_above_95:,}")


## 8. Department Analytics

**What:** For every department, compute employee count, average salary, and average performance score.

**Why:** Department-level breakdowns reveal how compensation and performance vary across teams, supporting workforce planning decisions.

**NumPy concepts:** `np.unique()` to find distinct department names, **boolean indexing** to isolate each department's records, and `np.mean()` / `np.sum()` for per-group aggregation.


In [ ]:
unique_departments = np.unique(department)

print("=" * 70)
print("DEPARTMENT ANALYTICS")
print("=" * 70)
print(f"{'Department':<15}{'Employees':<12}{'Avg Salary':<18}{'Avg Performance':<15}")
print("-" * 70)

for dept in unique_departments:
    dept_mask = department == dept
    dept_count = np.sum(dept_mask)
    dept_avg_salary = np.mean(salary[dept_mask])
    dept_avg_performance = np.mean(performance_score[dept_mask])

    print(
        f"{dept:<15}{dept_count:<12,}${dept_avg_salary:<17,.2f}{dept_avg_performance:<15.2f}"
    )


## 9. Performance Analytics

**What:** Identify the top and bottom performers by salary, top performers by performance score, and the highest attendance value.

**Why:** Ranking employees supports recognition programs, promotion decisions, and identifying outliers that may need review.

**NumPy concepts:** `np.argsort()` to obtain the indices that would sort an array, combined with slicing to extract the top/bottom N records.


In [ ]:
# argsort returns indices in ascending order; slice from the end for "top" values
salary_sorted_idx = np.argsort(salary)

top_10_salary_idx = salary_sorted_idx[-10:][::-1]
bottom_10_salary_idx = salary_sorted_idx[:10]

performance_sorted_idx = np.argsort(performance_score)
top_20_performance_idx = performance_sorted_idx[-20:][::-1]

highest_attendance_idx = np.argmax(attendance)

print("=" * 60)
print("TOP 10 HIGHEST SALARIES")
print("=" * 60)
for idx in top_10_salary_idx:
    print(f"ID {employee_id[idx]:>6} | {department[idx]:<12} | ${salary[idx]:>10,.2f}")

print("\n" + "=" * 60)
print("BOTTOM 10 SALARIES")
print("=" * 60)
for idx in bottom_10_salary_idx:
    print(f"ID {employee_id[idx]:>6} | {department[idx]:<12} | ${salary[idx]:>10,.2f}")

print("\n" + "=" * 60)
print("TOP 20 PERFORMANCE SCORES")
print("=" * 60)
for idx in top_20_performance_idx:
    print(f"ID {employee_id[idx]:>6} | {department[idx]:<12} | Score {performance_score[idx]:>3}")

print("\n" + "=" * 60)
print("HIGHEST ATTENDANCE")
print("=" * 60)
print(
    f"ID {employee_id[highest_attendance_idx]:>6} | "
    f"{department[highest_attendance_idx]:<12} | "
    f"Attendance {attendance[highest_attendance_idx]}%"
)


## 10. Bonus Calculation

**What:** Calculate each employee's bonus based on their performance score, using tiered rules:

- Performance ≥ 90 → Bonus = 20% of salary
- Performance ≥ 80 → Bonus = 10% of salary
- Otherwise → Bonus = 5% of salary

**Why:** Tiered, rule-based bonuses are a common real-world compensation pattern. Computing this over 100,000 records with a loop would be slow; NumPy lets us do it in a single vectorized pass.

**NumPy concepts:** `np.select()` for multi-condition vectorized assignment — no explicit loops required.


In [ ]:
# Vectorized multi-condition bonus calculation using np.select
conditions = [
    performance_score >= 90,
    performance_score >= 80,
]
bonus_rates = [0.20, 0.10]
default_rate = 0.05

bonus_percentage = np.select(conditions, bonus_rates, default=default_rate)
bonus = salary * bonus_percentage

print("=" * 60)
print("BONUS CALCULATION (Sample of First 10 Employees)")
print("=" * 60)
print(f"{'ID':<8}{'Performance':<13}{'Salary':<15}{'Bonus %':<10}{'Bonus':<12}")
print("-" * 60)
for i in range(10):
    print(
        f"{employee_id[i]:<8}{performance_score[i]:<13}"
        f"${salary[i]:<14,.2f}{bonus_percentage[i]*100:<9.0f}${bonus[i]:<11,.2f}"
    )


## 11. Final Salary Calculation

**What:** Compute each employee's final salary as `Salary + Bonus`.

**Why:** This produces the actual take-home compensation figure used in the final report.

**NumPy concepts:** Simple element-wise array addition — one of the most fundamental vectorized operations in NumPy.


In [ ]:
final_salary = salary + bonus

print("=" * 60)
print("FINAL SALARY (Sample of First 10 Employees)")
print("=" * 60)
print(f"{'ID':<8}{'Salary':<15}{'Bonus':<12}{'Final Salary':<15}")
print("-" * 60)
for i in range(10):
    print(
        f"{employee_id[i]:<8}${salary[i]:<14,.2f}${bonus[i]:<11,.2f}${final_salary[i]:<14,.2f}"
    )


## 12. Generate Final Employee Report

**What:** Combine all computed arrays into a single consolidated report array.

**Why:** A unified report is the natural end product of an analytics pipeline — one structure containing every relevant field per employee.

**NumPy concepts:** `np.column_stack()` to combine multiple 1D arrays into a single 2D array, column by column.


In [ ]:
final_report = np.column_stack((
    employee_id,
    department,
    salary,
    attendance,
    performance_score,
    bonus,
    final_salary,
))

print("=" * 100)
print("FINAL EMPLOYEE REPORT (First 10 Rows)")
print("=" * 100)
header = f"{'ID':<8}{'Department':<14}{'Salary':<14}{'Attendance':<12}{'Performance':<13}{'Bonus':<12}{'Final Salary':<14}"
print(header)
print("-" * 100)
for row in final_report[:10]:
    emp_id, dept, sal, att, perf, bon, fin_sal = row
    print(
        f"{emp_id:<8}{dept:<14}${float(sal):<13,.2f}{att:<12}{perf:<13}"
        f"${float(bon):<11,.2f}${float(fin_sal):<13,.2f}"
    )

print(f"\nFull report shape: {final_report.shape}")


## 13. NumPy vs Python Loop Performance Comparison

**What:** Recalculate the bonus for all 100,000 employees using two approaches — a traditional Python `for` loop and a NumPy vectorized operation — and compare their execution times.

**Why:** This is the core justification for using NumPy in data engineering workloads: vectorized operations run in optimized, compiled C code and avoid Python's per-element interpreter overhead, making them dramatically faster at scale.

**NumPy concepts:** `time.perf_counter()` for high-resolution timing, contrasted against `np.select()` vectorization.


In [ ]:
# --- Method 1: Traditional Python loop ---
start_loop = time.perf_counter()

loop_bonus = []
for i in range(NUM_EMPLOYEES):
    score = performance_score[i]
    sal = salary[i]
    if score >= 90:
        loop_bonus.append(sal * 0.20)
    elif score >= 80:
        loop_bonus.append(sal * 0.10)
    else:
        loop_bonus.append(sal * 0.05)

end_loop = time.perf_counter()
loop_duration = end_loop - start_loop

# --- Method 2: NumPy vectorized operation ---
start_vectorized = time.perf_counter()

vectorized_bonus = np.select(conditions, bonus_rates, default=default_rate) * salary

end_vectorized = time.perf_counter()
vectorized_duration = end_vectorized - start_vectorized

speedup = loop_duration / vectorized_duration

print("=" * 60)
print("PERFORMANCE BENCHMARK: PYTHON LOOP vs NUMPY")
print("=" * 60)
print(f"Python Loop Time        : {loop_duration:.5f} seconds")
print(f"NumPy Vectorized Time    : {vectorized_duration:.5f} seconds")
print(f"Speedup                  : {speedup:.1f}x faster with NumPy")


## 14. Conclusion

This notebook demonstrated a complete NumPy-only analytics workflow on a 100,000-record employee dataset:

- **Data generation** using `np.arange()`, `np.random.choice()`, `np.random.uniform()`, and `np.random.randint()`.
- **Descriptive statistics** for salary and attendance using `np.mean()`, `np.median()`, `np.std()`, `np.max()`, and `np.min()`.
- **Group-wise analytics** per department using `np.unique()` combined with boolean indexing.
- **Ranking and sorting** with `np.argsort()` to extract top and bottom performers.
- **Fully vectorized business logic** (tiered bonus calculation) using `np.select()` — no explicit loops.
- **Report generation** using `np.column_stack()` to consolidate results.
- **Performance benchmarking** proving that NumPy's vectorized operations significantly outperform equivalent Python loops at scale.

This workflow reflects patterns commonly used in real-world data engineering and analytics pipelines, where performance and code clarity both matter.
